In [1]:
import jsonlines
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import numpy as np
import nltk
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import regex as re
from collections import Counter

In [2]:
data_list = []

In [3]:
with jsonlines.open("data/train.jsonl", mode='r') as reader:
    for obj in reader:
        obj.pop('model')
        obj.pop('genre')
        obj.pop('id')
        data_list.append(obj)

In [4]:
len(data_list)

23707

In [5]:
df = pd.DataFrame(data_list)

In [6]:
print(df)

                                                    text  label
0      Duke Ellington, a titan of jazz, revolutionize...      1
1      I reflected on the shifting dynamics of media ...      1
2      In F. Scott Fitzgerald's "The Great Gatsby," t...      1
3      I still chuckle when I think about that time I...      1
4      Yoga, originating in ancient India, encompasse...      1
...                                                  ...    ...
23702  Antioch stands as a pivotal point in the narra...      1
23703  Margaret Cavendish’s The Blazing World stands ...      1
23704  In August of 1793, in a last-ditch effort to w...      0
23705  In interacting with Luciente and others, Conni...      0
23706  Meursault and Clamence, the protagonists from ...      1

[23707 rows x 2 columns]


In [7]:
tokens_all = [word_tokenize(sentence) for sentence in df.text]

In [8]:
# Removing all tokens consisting only of non-alphanumeric characters, that aren't inbetween words e.g. "battle-field",
# so any tokens such as: "''", "--", ".", "," etc. are removed, but "battle-field" is not removed. 
for sentence in tokens_all:
    while True:
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                sentence.remove(token) # For each token in a sentence, if the token consists only of 1 or more non-alphanumeric characters, remove it from the sentence.

        outs = [] # Creates an empty list. The list is reset to empty every iteration of the while-loop. 
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                outs.append(token) # This for-loop makes an additional check for any non-alphanumeric tokens, and appends then to the empty list 'outs'.

        if outs: # This checks whether the list 'outs' is empty or not. If it's not empty, then 'continue' is activated and the while-loop continues to iterate,
            continue # if the list is empty, then 'break' is activated, and the while-loop terminates, and the outer-most for-loop continues to the next 'sentence' in 'tokens_all'.
        else:
            break

In [9]:
word_list = [[word.lower() for word in sentence] for sentence in tokens_all]

In [10]:
len(word_list)

23707

In [11]:
words = []

In [12]:
for sentence in word_list:
    for word in sentence:
        words.append(word)

In [13]:
len(words)

14949644

In [14]:
vocab = sorted(set(words))

In [15]:
len(vocab)

122750

In [16]:
freq = Counter(words)

In [17]:
stop_words = set(stopwords.words('english'))

In [18]:
len(stop_words)

198

In [19]:
filtered_tokens = [word for word in words if word not in stop_words]

In [20]:
# This keeps the structure of a list with all datapoints in the train set, where each datapoint
# is in the form of a list. So it's a list with 23707 lists/data points with tokens.
# Each token has been converted to small characters with word.lower(). 
filtered_token_list = [[token for token in sentence if token not in stop_words] for sentence in word_list]
# Stop-words have now been removed as well. 

In [21]:
print(filtered_token_list[0])

['duke', 'ellington', 'titan', 'jazz', 'revolutionized', 'genre', 'innovative', 'compositions', 'showcasing', 'remarkable', 'ability', 'integrate', 'voice', 'instrumental', 'music', 'among', 'notable', 'figures', 'contributed', 'artistic', 'symphony', 'ivie', 'anderson', 'whose', 'scat', 'singing', 'mirrored', 'improvisational', 'prowess', 'musicians', 'like', 'nanton', 'ring', 'dem', 'bells', 'ellington', 'ingeniously', 'interweaves', 'scat', 'singing', 'dialogue', 'saxophones', 'creating', 'dynamic', 'call', 'response', 'underscores', 'vision', 'music', 'fluid', 'conversation', 'voices', 'instruments', 'ellington', 'consistently', 'emphasized', 'voice', 'instrument', 'equal', 'importance', 'traditional', 'brass', 'woodwinds', 'orchestrating', 'compositions', 'keen', 'ear', 'vocal', 'qualities', 'approach', 'vividly', 'demonstrated', 'mood', 'indigo', 'orchestration', 'skills', 'shine', 'non-traditional', 'chord', 'arrangements', 'transforming', 'piece', 'auditory', 'tapestry', 'mood'

In [22]:
filtered_token_strings = [" ".join(sentence) for sentence in filtered_token_list]

In [23]:
print(filtered_token_strings[:2])

["duke ellington titan jazz revolutionized genre innovative compositions showcasing remarkable ability integrate voice instrumental music among notable figures contributed artistic symphony ivie anderson whose scat singing mirrored improvisational prowess musicians like nanton ring dem bells ellington ingeniously interweaves scat singing dialogue saxophones creating dynamic call response underscores vision music fluid conversation voices instruments ellington consistently emphasized voice instrument equal importance traditional brass woodwinds orchestrating compositions keen ear vocal qualities approach vividly demonstrated mood indigo orchestration skills shine non-traditional chord arrangements transforming piece auditory tapestry mood color merely notes define ellington 's genius rather way orchestrates elements drawing unique talents band members like nanton hodges williams create symphony voice resonates authenticity purpose composition dusk exemplifies ellington 's adeptness capt

In [24]:
print(df)

                                                    text  label
0      Duke Ellington, a titan of jazz, revolutionize...      1
1      I reflected on the shifting dynamics of media ...      1
2      In F. Scott Fitzgerald's "The Great Gatsby," t...      1
3      I still chuckle when I think about that time I...      1
4      Yoga, originating in ancient India, encompasse...      1
...                                                  ...    ...
23702  Antioch stands as a pivotal point in the narra...      1
23703  Margaret Cavendish’s The Blazing World stands ...      1
23704  In August of 1793, in a last-ditch effort to w...      0
23705  In interacting with Luciente and others, Conni...      0
23706  Meursault and Clamence, the protagonists from ...      1

[23707 rows x 2 columns]


In [25]:
df['text'] = filtered_token_strings

In [26]:
print(df)

                                                    text  label
0      duke ellington titan jazz revolutionized genre...      1
1      reflected shifting dynamics media coverage per...      1
2      f. scott fitzgerald 's great gatsby use color ...      1
3      still chuckle think time bombed midterm exam m...      1
4      yoga originating ancient india encompasses div...      1
...                                                  ...    ...
23702  antioch stands pivotal point narrative early c...      1
23703  margaret cavendish blazing world stands pionee...      1
23704  august 1793 last-ditch effort win rebel leader...      0
23705  interacting luciente others connie struggles u...      0
23706  meursault clamence protagonists albert camus '...      1

[23707 rows x 2 columns]


In [27]:
len(words)

14949644

In [28]:
len(filtered_tokens)

8018027

In [29]:
filtered_vocab = sorted(set(filtered_tokens))

In [30]:
tags = pos_tag(filtered_tokens, tagset='universal')

In [31]:
tag_list = [pos_tag(sentence, tagset='universal') for sentence in filtered_token_list]

In [32]:
len(tag_list)

23707

In [33]:
sentence_tags = [[token[-1] for token in sentence] for sentence in tag_list]

In [34]:
print(sentence_tags[:2])

[['ADJ', 'NOUN', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'ADP', 'ADJ', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'NOUN', 'PRON', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'ADP', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'VERB', 'ADV', 'NOUN', 'VERB', 'VERB', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'VERB', 'ADV', 'VERB', 'NOUN', 'NOUN', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'ADJ', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'ADV', 'VERB', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'NOUN', 'ADV', 'VERB', 'ADJ', 'NOUN', 'PRT', 'NOUN', 'ADV', 'NOUN', 'VERB', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'VERB', 'NOUN', 'ADP', 'ADJ', 'NOUN', 'VERB', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'VERB', 'PRT', 'ADJ', 'VERB', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'NOUN', 'ADJ', 'VERB', 'A

In [35]:
unique_tags = set([tag[-1] for tag in tags])

In [36]:
len(unique_tags)

12

In [37]:
tags_freq = [dict(Counter(sentence)) for sentence in sentence_tags]

In [38]:
possible_tags = list(unique_tags)

In [39]:
print(possible_tags)

['.', 'CONJ', 'PRON', 'VERB', 'NOUN', 'ADJ', 'X', 'NUM', 'PRT', 'ADV', 'DET', 'ADP']


In [40]:
missing_tags = [{tag: 0 for tag in possible_tags if tag not in sentence} for sentence in tags_freq]

In [41]:
len(missing_tags)

23707

In [42]:
print(missing_tags[0])

{'.': 0, 'CONJ': 0, 'X': 0, 'NUM': 0, 'DET': 0}


In [43]:
print(tags_freq[0])

{'ADJ': 50, 'NOUN': 112, 'VERB': 47, 'ADP': 6, 'PRON': 1, 'ADV': 7, 'PRT': 3}


In [44]:
all_tags = []

for i in range(len(missing_tags)):
    tags = tags_freq[i]
    missing = missing_tags[i]
    tags.update(missing)
    all_tags.append(tags)

In [45]:
print(all_tags[0])

{'ADJ': 50, 'NOUN': 112, 'VERB': 47, 'ADP': 6, 'PRON': 1, 'ADV': 7, 'PRT': 3, '.': 0, 'CONJ': 0, 'X': 0, 'NUM': 0, 'DET': 0}


In [46]:
# Sorting the dict, 'example_tags', by making a dict comprehension and using sorted() with lambda
# inserted in the 'key'-argument of the sorted() function.
tags_sorted = [{k: v for k, v in sorted(sentence.items(), key=lambda item: item[0])} for sentence in all_tags]

# key=lambda item: item[0] is exactly the same as writing:
# key = def lambda(item):
#           return item[0]

# lambda is a throwaway function, just like _ is a throwaway variable.

In [47]:
print(tags_sorted[:2])

[{'.': 0, 'ADJ': 50, 'ADP': 6, 'ADV': 7, 'CONJ': 0, 'DET': 0, 'NOUN': 112, 'NUM': 0, 'PRON': 1, 'PRT': 3, 'VERB': 47, 'X': 0}, {'.': 0, 'ADJ': 59, 'ADP': 5, 'ADV': 14, 'CONJ': 0, 'DET': 1, 'NOUN': 137, 'NUM': 2, 'PRON': 0, 'PRT': 1, 'VERB': 83, 'X': 1}]


In [48]:
len(tags_sorted)

23707

In [49]:
tag_values = [list(sentence.values()) for sentence in tags_sorted]

In [50]:
len(tag_values)

23707

In [51]:
print(tag_values[:2])

[[0, 50, 6, 7, 0, 0, 112, 0, 1, 3, 47, 0], [0, 59, 5, 14, 0, 1, 137, 2, 0, 1, 83, 1]]


In [52]:
print(df)

                                                    text  label
0      duke ellington titan jazz revolutionized genre...      1
1      reflected shifting dynamics media coverage per...      1
2      f. scott fitzgerald 's great gatsby use color ...      1
3      still chuckle think time bombed midterm exam m...      1
4      yoga originating ancient india encompasses div...      1
...                                                  ...    ...
23702  antioch stands pivotal point narrative early c...      1
23703  margaret cavendish blazing world stands pionee...      1
23704  august 1793 last-ditch effort win rebel leader...      0
23705  interacting luciente others connie struggles u...      0
23706  meursault clamence protagonists albert camus '...      1

[23707 rows x 2 columns]


In [53]:
df['POS_vector'] = tag_values

In [54]:
print(df)

                                                    text  label  \
0      duke ellington titan jazz revolutionized genre...      1   
1      reflected shifting dynamics media coverage per...      1   
2      f. scott fitzgerald 's great gatsby use color ...      1   
3      still chuckle think time bombed midterm exam m...      1   
4      yoga originating ancient india encompasses div...      1   
...                                                  ...    ...   
23702  antioch stands pivotal point narrative early c...      1   
23703  margaret cavendish blazing world stands pionee...      1   
23704  august 1793 last-ditch effort win rebel leader...      0   
23705  interacting luciente others connie struggles u...      0   
23706  meursault clamence protagonists albert camus '...      1   

                                       POS_vector  
0        [0, 50, 6, 7, 0, 0, 112, 0, 1, 3, 47, 0]  
1       [0, 59, 5, 14, 0, 1, 137, 2, 0, 1, 83, 1]  
2       [0, 73, 7, 13, 0, 0, 141, 1, 0,

!!!-----------------DEV DATASET-----------------!!!

In [55]:
dev_data = []

with jsonlines.open("data/dev.jsonl", mode='r') as doc:
    for obj in doc:
        obj.pop('model')
        obj.pop('genre')
        obj.pop('id')
        dev_data.append(obj)


In [56]:
df_dev = pd.DataFrame(dev_data)
len(df_dev)

1795

In [57]:
print(df_dev)

                                                   text  label
0     Harry felt a sense of excitement as he walked ...      1
1     Squire Stretford was one of the good old sort ...      0
2     I look around and see the diversity of familie...      1
3     The young man returned home in a daze, his min...      1
4     The young officer's heart pounded in his chest...      1
...                                                 ...    ...
1790  Dora had the same yellow curly hair, in tight,...      0
1791  The air was thick with the hum of voices, the ...      1
1792  "My poor Dorothy! I do indeed feel for you."\n...      0
1793  Washington Post Staff Escapes Kabul Amidst Det...      1
1794  She at last acknowledged to Mr. Walker that sh...      0

[1795 rows x 2 columns]


In [58]:
dev_tokens = [word_tokenize(sentence) for sentence in df_dev.text]

In [59]:
# Removing all tokens consisting only of non-alphanumeric characters, that aren't inbetween words e.g. "battle-field",
# so any tokens such as: "''", "--", ".", "," etc. are removed, but "battle-field" is not removed. 
for sentence in dev_tokens:
    while True:
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                sentence.remove(token) # For each token in a sentence, if the token consists only of 1 or more non-alphanumeric characters, remove it from the sentence.

        outs = [] # Creates an empty list. The list is reset to empty every iteration of the while-loop. 
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                outs.append(token) # This for-loop makes an additional check for any non-alphanumeric tokens, and appends then to the empty list 'outs'.

        if outs: # This checks whether the list 'outs' is empty or not. If it's not empty, then 'continue' is activated and the while-loop continues to iterate,
            continue # if the list is empty, then 'break' is activated, and the while-loop terminates, and the outer-most for-loop continues to the next 'sentence' in 'tokens_all'.
        else:
            break

In [60]:
word_list_dev = [[word.lower() for word in sentence] for sentence in dev_tokens]

In [61]:
words_dev = []

In [62]:
for sentence in word_list_dev:
    for word in sentence:
        words_dev.append(word)

In [63]:
len(words_dev)

1109523

In [64]:
vocab_dev = sorted(set(words_dev))

In [65]:
len(vocab_dev)

41666

In [66]:
filtered_tokens_dev = [word for word in words_dev if word not in stop_words]

In [67]:
len(filtered_tokens_dev)

604723

In [68]:
# This keeps the structure of a list with all datapoints in the train set, where each datapoint
# is in the form of a list. So it's a list with 23707 lists/data points with tokens.
# Each token has been converted to small characters with word.lower(). 
filtered_token_list_dev = [[token for token in sentence if token not in stop_words] for sentence in word_list_dev]
# Stop-words have now been removed as well. 

In [69]:
filtered_token_strings_dev = [" ".join(sentence) for sentence in filtered_token_list_dev]

In [70]:
df_dev['text'] = filtered_token_strings_dev

In [71]:
print(df_dev)

                                                   text  label
0     harry felt sense excitement walked towards new...      1
1     squire stretford one good old sort country gen...      0
2     look around see diversity families life single...      1
3     young man returned home daze mind swirling con...      1
4     young officer 's heart pounded chest followed ...      1
...                                                 ...    ...
1790  dora yellow curly hair tight frizzly rings hea...      0
1791  air thick hum voices rustle silken gowns faint...      1
1792  poor dorothy indeed feel knew saw headless hor...      0
1793  washington post staff escapes kabul amidst det...      1
1794  last acknowledged mr. walker could account twe...      0

[1795 rows x 2 columns]


In [72]:
tags_dev = pos_tag(filtered_tokens_dev, tagset='universal')

In [73]:
len(tags_dev)

604723

In [74]:
tag_list_dev = [pos_tag(sentence, tagset='universal') for sentence in filtered_token_list_dev]

In [75]:
len(tag_list_dev)

1795

In [76]:
sentence_tags_dev = [[token[-1] for token in sentence] for sentence in tag_list_dev]

In [77]:
unique_tags_dev = set([tag[-1] for tag in tags_dev])

In [78]:
len(unique_tags_dev)

12

In [79]:
tags_freq_dev = [dict(Counter(sentence)) for sentence in sentence_tags_dev]

In [80]:
missing_tags_dev = [{tag: 0 for tag in possible_tags if tag not in sentence} for sentence in tags_freq_dev]

In [81]:
len(missing_tags_dev)

1795

In [82]:
all_tags_dev = []

for i in range(len(missing_tags_dev)):
    tags_dev = tags_freq_dev[i]
    missing_dev = missing_tags_dev[i]
    tags_dev.update(missing_dev)
    all_tags_dev.append(tags_dev)

In [83]:
# Sorting the dict, 'example_tags', by making a dict comprehension and using sorted() with lambda
# inserted in the 'key'-argument of the sorted() function.
tags_sorted_dev = [{k: v for k, v in sorted(sentence.items(), key=lambda item: item[0])} for sentence in all_tags_dev]

# key=lambda item: item[0] is exactly the same as writing:
# key = def lambda(item):
#           return item[0]

# lambda is a throwaway function, just like _ is a throwaway variable.

In [84]:
len(tags_sorted_dev)

1795

In [85]:
tag_values_dev = [list(sentence.values()) for sentence in tags_sorted_dev]

In [86]:
len(tag_values_dev)

1795

In [87]:
print(tag_values_dev[:2])

[[0, 49, 8, 15, 0, 1, 134, 4, 0, 10, 81, 0], [0, 68, 3, 27, 0, 0, 144, 5, 0, 1, 95, 0]]


In [88]:
df_dev['POS_vector'] = tag_values_dev

In [89]:
print(df_dev)

                                                   text  label  \
0     harry felt sense excitement walked towards new...      1   
1     squire stretford one good old sort country gen...      0   
2     look around see diversity families life single...      1   
3     young man returned home daze mind swirling con...      1   
4     young officer 's heart pounded chest followed ...      1   
...                                                 ...    ...   
1790  dora yellow curly hair tight frizzly rings hea...      0   
1791  air thick hum voices rustle silken gowns faint...      1   
1792  poor dorothy indeed feel knew saw headless hor...      0   
1793  washington post staff escapes kabul amidst det...      1   
1794  last acknowledged mr. walker could account twe...      0   

                                        POS_vector  
0       [0, 49, 8, 15, 0, 1, 134, 4, 0, 10, 81, 0]  
1        [0, 68, 3, 27, 0, 0, 144, 5, 0, 1, 95, 0]  
2        [0, 53, 4, 27, 0, 0, 102, 0, 0, 2, 50, 

In [90]:
# Examine your Vocabulary (e.g., sort by frequency, alphabet, etc.).
#    a. What kinds of words do you see?
#    b. Does your text normalization seem to work well? Why or why not?
#    c. Are there marked differences between human_Vocab and
#       machine_Vocab?
#    d. What % of tokens were seen in train but not dev?

In [91]:
filtered_vocab_dev = sorted(set(filtered_tokens_dev))

In [92]:
len(filtered_vocab_dev)

41514

In [93]:
len(filtered_vocab)

122595

In [94]:
missing_tokens_dev = [token for token in filtered_vocab_dev if token not in filtered_vocab]

In [95]:
len(missing_tokens_dev)

3743

In [96]:
human_train_df = df

In [97]:
print(human_train_df)

                                                    text  label  \
0      duke ellington titan jazz revolutionized genre...      1   
1      reflected shifting dynamics media coverage per...      1   
2      f. scott fitzgerald 's great gatsby use color ...      1   
3      still chuckle think time bombed midterm exam m...      1   
4      yoga originating ancient india encompasses div...      1   
...                                                  ...    ...   
23702  antioch stands pivotal point narrative early c...      1   
23703  margaret cavendish blazing world stands pionee...      1   
23704  august 1793 last-ditch effort win rebel leader...      0   
23705  interacting luciente others connie struggles u...      0   
23706  meursault clamence protagonists albert camus '...      1   

                                       POS_vector  
0        [0, 50, 6, 7, 0, 0, 112, 0, 1, 3, 47, 0]  
1       [0, 59, 5, 14, 0, 1, 137, 2, 0, 1, 83, 1]  
2       [0, 73, 7, 13, 0, 0, 141, 1, 0,

In [98]:
human_train_df.drop(df[df['label']==1].index)

,text,label,POS_vector
13,private bank corruption ultimately drove natio...,0,"[0, 60, 7, 14, 0, 2, 131, 9, 0, 0, 60, 0]"
22,idea behind dhutanga would help monks shed wor...,0,"[0, 59, 3, 22, 0, 1, 127, 3, 0, 0, 56, 0]"
25,approach corner hallway door classroom front o...,0,"[0, 47, 6, 27, 0, 0, 91, 3, 0, 0, 58, 0]"
47,may feel free point essay desire reviewers lea...,0,"[0, 56, 6, 24, 0, 2, 103, 2, 1, 8, 47, 0]"
49,many students especially younger innate desire...,0,"[0, 49, 3, 21, 1, 0, 101, 4, 0, 0, 58, 0]"
...,...,...,...
23693,years students challenged academically within ...,0,"[0, 60, 9, 41, 1, 2, 138, 6, 0, 0, 51, 0]"
23697,final concept crucial exploration titusville f...,0,"[0, 52, 4, 18, 0, 0, 153, 1, 0, 0, 65, 0]"
23699,calculus mathematics also teach problem solvin...,0,"[0, 75, 11, 28, 1, 3, 131, 4, 0, 1, 54, 0]"
23704,august 1793 last-ditch effort win rebel leader...,0,"[0, 74, 6, 19, 0, 0, 149, 14, 0, 0, 66, 0]"


In [99]:
llm_train_df = df

In [100]:
llm_train_df.drop(df[df['label']==0].index)

,text,label,POS_vector
0,duke ellington titan jazz revolutionized genre...,1,"[0, 50, 6, 7, 0, 0, 112, 0, 1, 3, 47, 0]"
1,reflected shifting dynamics media coverage per...,1,"[0, 59, 5, 14, 0, 1, 137, 2, 0, 1, 83, 1]"
2,f. scott fitzgerald 's great gatsby use color ...,1,"[0, 73, 7, 13, 0, 0, 141, 1, 0, 7, 55, 0]"
3,still chuckle think time bombed midterm exam m...,1,"[0, 45, 11, 13, 0, 1, 89, 4, 1, 9, 65, 0]"
4,yoga originating ancient india encompasses div...,1,"[0, 76, 7, 16, 0, 0, 163, 5, 0, 5, 67, 1]"
...,...,...,...
23700,recent debate surrounding bump stocks new regu...,1,"[0, 54, 4, 24, 0, 0, 111, 1, 0, 2, 49, 1]"
23701,karen driving interstate 77 windshield wipers ...,1,"[0, 63, 4, 30, 0, 1, 139, 5, 0, 1, 94, 0]"
23702,antioch stands pivotal point narrative early c...,1,"[0, 49, 3, 9, 0, 0, 106, 0, 0, 1, 47, 0]"
23703,margaret cavendish blazing world stands pionee...,1,"[0, 94, 1, 16, 0, 0, 146, 0, 0, 0, 57, 0]"
